### Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:

1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [3]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

from langchain_groq import ChatGroq

groq_llm = ChatGroq(model="qwen/qwen3.8-27b")

response = groq_llm.invoke("Tell about Mumbai")
print(response.content)


Mumbai, often referred to as the "City of Dreams" or the "Gateway of India," is the capital of the Indian state of Maharashtra and the most populous city in India. It is a global financial center, the center of the Indian film industry (Bollywood), and a major port city.

Here’s a comprehensive overview of Mumbai:

### 1. **Geography & Layout**
- **Location**: Situated on the west coast of India, along the Arabian Sea.
- **Islands**: The city is built on seven islands (formerly separate entities) that were joined together through land reclamation and bridges. These include Colaba, Worli, Girgaon, Mahim, Malabar Hill, and others.
- **Climate**: Tropical monsoon climate – hot and humid year-round, with a wet monsoon season (June–September) and a dry season (October–May).

### 2. **Economic Hub**
- **Financial Capital**: Mumbai is the commercial and financial capital of India. It is home to:
  - The Bombay Stock Exchange (BSE) and National Stock Exchange (NSE).
  - Major headquarters of I

In [4]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """ Get the Weather of Location"""
    return "It is 32 C and sunny in {location}"

model_with_tools=groq_llm.bind_tools([get_weather])


In [6]:
response = model_with_tools.invoke("What is weather in Mumbai?")
print(response)
for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'tool_calls': [{'id': 'vv1wdec8g', 'function': {'arguments': '{"location":"Mumbai"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 275, 'total_tokens': 302, 'completion_time': 0.073966379, 'completion_tokens_details': None, 'prompt_time': 0.022688059, 'prompt_tokens_details': None, 'queue_time': 0.048518571, 'total_time': 0.096654438}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_4560dae850', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a0b02e-705b-76e3-b2a1-a7e29330c9f6-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Mumbai'}, 'id': 'vv1wdec8g', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 275, 'output_tokens': 27, 'total_tokens': 302}
Tool: get_weather
Args: {'location': 'Mumbai'}


### Tool Execution Steps


In [11]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Mumbai?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)


The current weather in Mumbai is **32°C (90°F)** and **sunny**.


In [10]:
messages

[{'role': 'user', 'content': "What's the weather in Mumbai?"},
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'q3wk4gg1s', 'function': {'arguments': '{"location":"Mumbai"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 276, 'total_tokens': 303, 'completion_time': 0.074112431, 'completion_tokens_details': None, 'prompt_time': 0.022760327, 'prompt_tokens_details': None, 'queue_time': 0.056409252, 'total_time': 0.096872758}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_4560dae850', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0b035-0724-7351-9f1f-c507a09a7706-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Mumbai'}, 'id': 'q3wk4gg1s', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 276, 'output_tokens': 27, 'total_tokens': 303}),
 ToolMessage(content='It is 32 C 